In [1]:
import os
from pathlib import Path
import re
import json
from dotenv import load_dotenv
import pandas as pd
from typing import Optional, TypedDict, Any

from pydantic import BaseModel, Field

from langchain_core.documents import Document

import json
import os
from pathlib import Path
import re
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from sqlmodel import Session

from langchain_openai import ChatOpenAI, OpenAIEmbeddings


# # ROOT_DIR = Path(__file__).resolve().parent.parent.parent
# ROOT_DIR = Path(__file__).resolve().parents[2]
# print("ROOT DIR: ",ROOT_DIR)

# ENV_PATH = ROOT_DIR / ".env"
# APP_DIR = ROOT_DIR / "app"
# load_dotenv(ENV_PATH)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
if not os.environ.get("OPENAI_API_KEY"):
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# df = pd.read_csv("data/movies_plots.csv")


In [28]:
df = pd.read_csv("data/without_chunks.csv")
# df = pd.read_csv("F:\\Users\\User\\Desktop\\data\\chroma_clean.csv")
# df[df['doc_type'].isin(['full_plot_semantic_query', 'summary_semantic_query'])].info()


In [29]:
# df[~df['doc_type'].isin(['full_plot', 'summary'])].head()
# df_rat = pd.read_csv("data/movie_ratings.csv")  
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21972 entries, 0 to 21971
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   chroma_id     21972 non-null  object
 1   movie_id      21972 non-null  int64 
 2   source_movie  5498 non-null   object
 3   page_content  21972 non-null  object
 4   genres        21972 non-null  object
 5   director      21972 non-null  object
 6   title         21972 non-null  object
 7   year          21972 non-null  int64 
 8   doc_type      21972 non-null  object
dtypes: int64(2), object(7)
memory usage: 1.5+ MB


In [14]:
df_mr = df_rat[df_rat['MovieID'].isin(df['movie_id'])] 
df_mr.head()


,RaterID,MovieID,Rating,Timestamp
0,36955,21,3.0,1995-01-09 11:46:49
1,35435,45,5.0,1996-01-29 00:00:00
2,35435,21,5.0,1996-01-29 00:00:00
3,35139,52,4.0,1996-01-29 00:00:00
4,35139,50,5.0,1996-01-29 00:00:00


In [ ]:
(df_rat['RaterID'].value_counts()<100).value_counts()
# (df_rat['MovieID'].value_counts()<100).info()
# df40 = df_rat['MovieID'].value_counts()<40
# df40.index.tolist()[-1]
# df40.index[-1]

filt_df = df_rat.groupby('RaterID').filter(lambda x: len(x) < 100)
# df_rat.info()
# df_rat

RaterID  MovieID  Rating  Timestamp          
1        122      5.0     1996-08-02 11:24:06    1
         185      5.0     1996-08-02 10:58:45    1
         292      5.0     1996-08-02 10:57:01    1
         316      5.0     1996-08-02 10:56:32    1
         355      5.0     1996-08-02 11:14:34    1
                                                ..
71567    2107     1.0     1998-12-02 06:35:53    1
         2126     2.0     1998-12-03 01:39:03    1
         2294     5.0     1998-12-02 05:52:48    1
         2338     2.0     1998-12-02 05:53:36    1
         2384     2.0     1998-12-02 05:56:13    1
Name: count, Length: 2098790, dtype: int64

In [123]:
(filt_df['MovieID'].value_counts()<100).value_counts()


count
True     3264
False    1850
Name: count, dtype: int64

In [76]:
# (df_rat['MovieID'].value_counts()<40).index
counts  = df_rat['MovieID'].value_counts()
# filtered_df = df_rat[df_rat['MovieID'].map(counts) < 100]
counts

MovieID
296      34864
356      34457
593      33668
480      32631
318      31126
         ...  
8038         2
25832        2
3383         2
26067        1
37957        1
Name: count, Length: 5497, dtype: int64

In [85]:
hf = df_rat[df_rat['MovieID'].isin(df40.index.tolist())]
hf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7910635 entries, 0 to 7910634
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   RaterID    int64  
 1   MovieID    int64  
 2   Rating     float64
 3   Timestamp  object 
dtypes: float64(1), int64(2), object(1)
memory usage: 241.4+ MB


In [ ]:
# df.sort_values(by=['movie_id','doc_type'])
# filtered_df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 68848 entries, 626 to 7910627
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   RaterID    68848 non-null  int64  
 1   MovieID    68848 non-null  int64  
 2   Rating     68848 non-null  float64
 3   Timestamp  68848 non-null  object 
dtypes: float64(1), int64(2), object(1)
memory usage: 2.6+ MB


In [16]:
import chromadb

CHROMA_DIR = "app/chromaras"

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection("movies")

print("Count:", collection.count())

Count: 4000


In [17]:
def chroma_documents_to_df(collection, batch_size=1000):
    total = collection.count()
    rows = []

    for offset in range(0, total, batch_size):
        result = collection.get(
            limit=batch_size,
            offset=offset,
            include=["documents", "metadatas"]  # ΟΧΙ embeddings
        )

        for chroma_id, document, metadata in zip(
            result["ids"],
            result["documents"],
            result["metadatas"]
        ):
            metadata = metadata or {}

            rows.append({
                "chroma_id": chroma_id,
                "page_content": document,
                **metadata
            })

        print(f"Loaded {min(offset + batch_size, total)} / {total}")

    return pd.DataFrame(rows)

In [18]:
chroma_df = chroma_documents_to_df(collection)

# print(chroma_df.shape)
# chroma_df[chroma_df['doc_type'].isin(['full_plot', 'summary'])].info()


Loaded 1000 / 4000
Loaded 2000 / 4000
Loaded 3000 / 4000
Loaded 4000 / 4000


In [20]:
chroma_df


,chroma_id,page_content,genre_animation,year,genre_children,movie_id,genres,source_movie,director,genre_adventure,...,genre_thriller,genre_horror,genre_mystery,genre_sci_fi,genre_imax,genre_war,genre_musical,genre_film_noir,genre_western,genre_documentary
0,0_full_plot,In a world where toys are living things who pr...,True,1995,True,1,Adventure|Animation|Children|Comedy|Fantasy,,John Lasseter,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1_summary,"In ""Toy Story,"" toys come to life when humans ...",True,1995,True,1,Adventure|Animation|Children|Comedy|Fantasy,1|Toy Story,John Lasseter,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2_full_plot,"In 1869, near Brantford, New Hampshire, two br...",NaN,1995,True,2,Adventure|Children|Fantasy,,Joe Johnston,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3_summary,"In ""Jumanji,"" a board game unleashes chaos whe...",NaN,1995,True,2,Adventure|Children|Fantasy,2|Jumanji,Joe Johnston,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4_full_plot,The feud between Max (Walter Matthau) and John...,NaN,1995,NaN,3,Comedy|Romance,,Howard Deutch,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,3995_summary,"In ""Montana,"" professional hit woman Claire, p...",NaN,1998,NaN,3184,Action|Comedy|Crime|Drama,3184|Montana,Jennifer Leitzes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3996,3996_full_plot,Set on the fictional San Piedro Island in the ...,NaN,1999,NaN,3185,Drama,,Scott Hicks,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3997,3997_summary,Set in 1950 on the fictional San Piedro Island...,NaN,1999,NaN,3185,Drama,3185|Snow Falling on Cedars,Scott Hicks,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3998,3998_full_plot,After 18-year-old Susanna Kaysen has a nervous...,NaN,1999,NaN,3186,Drama,,James Mangold,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# chroma_df = chroma_df[['chroma_id','movie_id', 'source_movie','page_content','genres','director','title','year','doc_type']]
chroma_clean_df = chroma_df[['chroma_id','movie_id', 'source_movie','page_content','genres','director','title','year','doc_type']]

In [7]:
# chroma_clean_df[chroma_clean_df['doc_type']=='full_plot_semantic_query'].info()
chroma_clean_df

,chroma_id,movie_id,source_movie,page_content,genres,director,title,year,doc_type
0,e9f51f0b-425c-4a3f-a9e2-debe8c76d3eb,1,NaN,In a world where toys are living things who pr...,Adventure|Animation|Children|Comedy|Fantasy,John Lasseter,Toy Story,1995,full_plot
1,d7be0b17-668b-4d9b-b78b-3d96850bcc80,1,1|Toy Story,"In ""Toy Story,"" toys come to life when humans ...",Adventure|Animation|Children|Comedy|Fantasy,John Lasseter,Toy Story,1995,summary
2,2719c1d6-0f08-4c53-8395-88978ab6cb66,2,NaN,"In 1869, near Brantford, New Hampshire, two br...",Adventure|Children|Fantasy,Joe Johnston,Jumanji,1995,full_plot
3,788d850a-2dab-420f-9736-666f88805b8d,2,2|Jumanji,"In ""Jumanji,"" a board game unleashes chaos whe...",Adventure|Children|Fantasy,Joe Johnston,Jumanji,1995,summary
4,0990e7cb-2024-48e3-bb75-7530d5dd7912,3,NaN,The feud between Max (Walter Matthau) and John...,Comedy|Romance,Howard Deutch,Grumpier Old Men,1995,full_plot
...,...,...,...,...,...,...,...,...,...
52281,movie_38294_chunk_par_2,38294,,"A villager, recently baptized and thus now una...",Action|Adventure|Drama|Fantasy,Sturla Gunnarsson,Beowulf & Grendel,2005,chunk_par
52282,movie_38294_chunk_par_3,38294,,The Danes are later attacked by Grendel's moth...,Action|Adventure|Drama|Fantasy,Sturla Gunnarsson,Beowulf & Grendel,2005,chunk_par
52283,movie_38388_chunk_par_0,38388,,Santiago Muñez is a skilled footballer. The so...,Drama,Danny Cannon,Goal!,2005,chunk_par
52284,movie_38388_chunk_par_1,38388,,Glen warmly welcomes Santiago to his home and ...,Drama,Danny Cannon,Goal!,2005,chunk_par


In [8]:
chroma_clean_df.to_csv('data/movies_data.csv', index=False, encoding='utf-8-sig')

In [16]:
# chroma_df["doc_type"] = chroma_df["doc_type"].astype(str).str.strip()

chroma_clean_df[chroma_df["doc_type"] == 'full_plot']
# chroma_clean_df.info()

,chroma_id,movie_id,source_movie,page_content,genres,director,title,year,doc_type
0,e9f51f0b-425c-4a3f-a9e2-debe8c76d3eb,1,NaN,In a world where toys are living things who pr...,Adventure|Animation|Children|Comedy|Fantasy,John Lasseter,Toy Story,1995,full_plot
2,2719c1d6-0f08-4c53-8395-88978ab6cb66,2,NaN,"In 1869, near Brantford, New Hampshire, two br...",Adventure|Children|Fantasy,Joe Johnston,Jumanji,1995,full_plot
4,0990e7cb-2024-48e3-bb75-7530d5dd7912,3,NaN,The feud between Max (Walter Matthau) and John...,Comedy|Romance,Howard Deutch,Grumpier Old Men,1995,full_plot
6,758ceb14-cf70-485f-9a8c-7ff4a9a32192,4,NaN,"""Friends are the People who let you be yoursel...",Comedy|Drama|Romance,Forest Whitaker,Waiting to Exhale,1995,full_plot
8,65bfa69a-90ea-4d6a-93ad-18fe016cf9e6,5,NaN,The film begins five years after the events of...,Comedy,Charles Shyer,Father of the Bride Part II,1995,full_plot
...,...,...,...,...,...,...,...,...,...
10986,movie_37957_full_plot,37957,,"Meat merchant Oleg, prostitute Marina, and pia...",Drama,Ilya Khrzhanovsky,4,2005,full_plot
10988,movie_38038_full_plot,38038,,Tottington Hall's annual giant vegetable compe...,Adventure|Animation|Children|Comedy,"Steve Box, Nick Park",Wallace & Gromit: The Curse of the Were-Rabbit,2005,full_plot
10990,movie_38061_full_plot,38061,,"At a Los Angeles party, Harry Lockhart recount...",Action|Comedy|Mystery|Thriller,Shane Black,Kiss Kiss Bang Bang,2005,full_plot
10992,movie_38294_full_plot,38294,,"In 500 A.D., Hrothgar, king of Denmark, and a ...",Action|Adventure|Drama|Fantasy,Sturla Gunnarsson,Beowulf & Grendel,2005,full_plot


In [21]:
full_plot_df = chroma_df[chroma_df["doc_type"] == "full_plot"].copy()
summary_df = chroma_df[chroma_df["doc_type"] == "summary"].copy()

summary_movie_ids = set(summary_df["movie_id"])

missing_summary_df = full_plot_df[
    ~full_plot_df["movie_id"].isin(summary_movie_ids)
].copy()

In [56]:
# missing_summary_df = missing_summary_df.drop(columns = ['source_movie'] )
missing_summary_df.to_csv('C:\\Users\\User\\Desktop\\missing_summary.csv', index=False, encoding='utf-8-sig')



,chroma_id,movie_id,source_movie,page_content,genres,director,title,year,doc_type
0,e9f51f0b-425c-4a3f-a9e2-debe8c76d3eb,1,NaN,In a world where toys are living things who pr...,Adventure|Animation|Children|Comedy|Fantasy,John Lasseter,Toy Story,1995,full_plot
1,d7be0b17-668b-4d9b-b78b-3d96850bcc80,1,1|Toy Story,"In ""Toy Story,"" toys come to life when humans ...",Adventure|Animation|Children|Comedy|Fantasy,John Lasseter,Toy Story,1995,summary
2,2719c1d6-0f08-4c53-8395-88978ab6cb66,2,NaN,"In 1869, near Brantford, New Hampshire, two br...",Adventure|Children|Fantasy,Joe Johnston,Jumanji,1995,full_plot
3,788d850a-2dab-420f-9736-666f88805b8d,2,2|Jumanji,"In ""Jumanji,"" a board game unleashes chaos whe...",Adventure|Children|Fantasy,Joe Johnston,Jumanji,1995,summary
4,0990e7cb-2024-48e3-bb75-7530d5dd7912,3,NaN,The feud between Max (Walter Matthau) and John...,Comedy|Romance,Howard Deutch,Grumpier Old Men,1995,full_plot
...,...,...,...,...,...,...,...,...,...
10941,NaN,31685,31685|Hitch,"In ""Hitch,"" Alex ""Hitch"" Hitchens is a success...",Comedy|Romance,Andy Tennant,Hitch,2005,summary
10942,NaN,32153,32153|Once Upon a Forest,"In ""Once Upon a Forest,"" four young animals—Ab...",Adventure|Animation|Children|Fantasy,Charles Grosvenor,Once Upon a Forest,1993,summary
10943,NaN,32882,32882|The Big Store,"In ""The Big Store,"" the death of Hiram Phelps ...",Comedy,Charles Reisner,The Big Store,1941,summary
10944,NaN,34326,34326|Last Days,"In ""Last Days,"" a young musician named Blake e...",Drama,Gus Van Sant,Last Days,2005,summary


In [46]:
missing_summary_df[missing_summary_df['doc_type'] == "summary"]

,chroma_id,movie_id,page_content,genres,director,title,year,doc_type


In [47]:
generated_summaries_df = pd.read_csv("data/missing_summaries_generated.csv")
generated_summaries_df

,chroma_id,movie_id,source_movie,page_content,genres,director,title,year,doc_type
0,NaN,1956,1956|Ordinary People,"In ""Ordinary People,"" the Jarrett family strug...",Drama,Robert Redford,Ordinary People,1980,summary
1,NaN,1957,1957|Chariots of Fire,"Set in the early 20th century, ""Chariots of Fi...",Drama,Hugh Hudson,Chariots of Fire,1981,summary
2,NaN,1958,1958|Terms of Endearment,"""Terms of Endearment"" follows the complex rela...",Comedy|Drama,James L. Brooks,Terms of Endearment,1983,summary
3,NaN,1959,1959|Out of Africa,"In ""Out of Africa,"" set in early 20th century ...",Drama|Romance,Sydney Pollack,Out of Africa,1985,summary
4,NaN,1960,1960|The Last Emperor,"""The Last Emperor"" chronicles the life of Puyi...",Drama,Bernardo Bertolucci,The Last Emperor,1987,summary
5,NaN,1961,1961|Rain Man,"In ""Rain Man,"" Charlie Babbitt discovers he ha...",Drama,Barry Levinson,Rain Man,1988,summary
6,NaN,1962,1962|Driving Miss Daisy,"Set in Atlanta, Georgia, from 1948 to the earl...",Drama,Bruce Beresford,Driving Miss Daisy,1989,summary
7,NaN,1963,1963|Take the Money and Run,"""Take the Money and Run"" is a comedic crime fi...",Comedy|Crime,Woody Allen,Take the Money and Run,1969,summary
8,NaN,7743,7743|Explorers,"In ""Explorers"" (1985), young teen Ben Crandall...",Adventure|Children|Fantasy|Sci-Fi,Joe Dante,Explorers,1985,summary
9,NaN,8057,8057|Sweet Bird of Youth,"In ""Sweet Bird of Youth,"" handsome young Chanc...",Drama,Richard Brooks,Sweet Bird of Youth,1962,summary


In [49]:
with_summary_df = pd.concat(
    [chroma_df, generated_summaries_df],
    ignore_index=True
)

print(with_summary_df["doc_type"].value_counts())



doc_type
full_plot    5473
summary      5473
Name: count, dtype: int64


In [167]:
with_summary_df
# .to_csv('C:\\Users\\User\\Desktop\\dev\\fastApi_heroes\\data\\with_summary.csv', index=False, encoding='utf-8-sig')
# # generated_summaries_df

,chroma_id,movie_id,source_movie,page_content,genres,director,title,year,doc_type
0,e9f51f0b-425c-4a3f-a9e2-debe8c76d3eb,1,NaN,In a world where toys are living things who pr...,Adventure|Animation|Children|Comedy|Fantasy,John Lasseter,Toy Story,1995,full_plot
1,d7be0b17-668b-4d9b-b78b-3d96850bcc80,1,1|Toy Story,"In ""Toy Story,"" toys come to life when humans ...",Adventure|Animation|Children|Comedy|Fantasy,John Lasseter,Toy Story,1995,summary
2,2719c1d6-0f08-4c53-8395-88978ab6cb66,2,NaN,"In 1869, near Brantford, New Hampshire, two br...",Adventure|Children|Fantasy,Joe Johnston,Jumanji,1995,full_plot
3,788d850a-2dab-420f-9736-666f88805b8d,2,2|Jumanji,"In ""Jumanji,"" a board game unleashes chaos whe...",Adventure|Children|Fantasy,Joe Johnston,Jumanji,1995,summary
4,0990e7cb-2024-48e3-bb75-7530d5dd7912,3,NaN,The feud between Max (Walter Matthau) and John...,Comedy|Romance,Howard Deutch,Grumpier Old Men,1995,full_plot
...,...,...,...,...,...,...,...,...,...
10941,NaN,31685,31685|Hitch,"In ""Hitch,"" Alex ""Hitch"" Hitchens is a success...",Comedy|Romance,Andy Tennant,Hitch,2005,summary
10942,NaN,32153,32153|Once Upon a Forest,"In ""Once Upon a Forest,"" four young animals—Ab...",Adventure|Animation|Children|Fantasy,Charles Grosvenor,Once Upon a Forest,1993,summary
10943,NaN,32882,32882|The Big Store,"In ""The Big Store,"" the death of Hiram Phelps ...",Comedy,Charles Reisner,The Big Store,1941,summary
10944,NaN,34326,34326|Last Days,"In ""Last Days,"" a young musician named Blake e...",Drama,Gus Van Sant,Last Days,2005,summary


In [127]:
# movie_texts_df = chroma_df.pivot_table(
#     index=["movie_id", "title", "year", "genres"],
#     columns="doc_type",
#     values="page_content",
#     aggfunc="first"
# ).reset_index()

# movie_texts_df.columns.name = None

# movie_texts_df.head()
# df_basic.info()
current_movie_ids =  set(with_summary_df[(~with_summary_df['page_content'].isna() & (with_summary_df['doc_type']=='full_plot'))]['movie_id'])

In [134]:
missings = missing_movies_df

In [158]:
missings

,movie_id,title,genres,year,director,genre,page_content,doc_type
9216,36517,The Constant Gardener,Drama|Thriller,2005,Fernando Meirelles,political thriller,"Justin Quayle (Ralph Fiennes), a shy, low-leve...",full_plot
9219,36525,Just Like Heaven,Comedy|Romance,2005,Mark Waters,comedy,"Elizabeth Masterson (Witherspoon), a young eme...",full_plot
9220,36527,Proof,Drama,2005,John Madden,drama,The plot alternates between events immediately...,full_plot
9221,36529,Lord of War,Action|Crime|Drama|Romance|Thriller|War,2005,Andrew Niccol,crime drama,"In the early 1980s, Yuri Orlov (Nicolas Cage),...",full_plot
9224,36535,Everything Is Illuminated,Adventure|Comedy|Drama,2005,Liev Schreiber,drama,"Jonathan Safran Foer (Elijah Wood), a young Am...",full_plot
...,...,...,...,...,...,...,...,...
10673,65025,Double Dynamite,Comedy|Musical,1951,Irving Cummings,musical comedy,Meek California Fidelity Trust teller Johnny D...,full_plot
10676,65088,Bedtime Stories,Adventure|Children|Comedy,2008,Adam Shankman,"family, fantasy",Skeeter Bronson (Adam Sandler) is a hotel main...,full_plot
10677,65091,Manhattan Melodrama,Crime|Drama|Romance,1934,George Cukor,"drama, romance","On June 15, 1904, the ship General Slocum catc...",full_plot
10678,65126,Choke,Comedy|Drama,2008,Clark Gregg,comedy-drama,Victor Mancini is a sex addict who works as a ...,full_plot


In [138]:

last_summaries = pd.read_csv("data/last_summaries_complete.csv")

In [145]:
last_summaries

,movie_id,source_movie,page_content,genres,director,title,year,doc_type
0,36517,36517|The Constant Gardener,"In ""The Constant Gardener,"" Justin Quayle, a B...",Drama|Thriller,Fernando Meirelles,The Constant Gardener,2005,summary
1,36525,36525|Just Like Heaven,"In ""Just Like Heaven,"" Elizabeth Masterson, a ...",Comedy|Romance,Mark Waters,Just Like Heaven,2005,summary
2,36527,36527|Proof,"In ""Proof"" (2005), Catherine, a mathematician ...",Drama,John Madden,Proof,2005,summary
3,36529,36529|Lord of War,"In ""Lord of War,"" set in the early 1980s, Ukra...",Action|Crime|Drama|Romance|Thriller|War,Andrew Niccol,Lord of War,2005,summary
4,36535,36535|Everything Is Illuminated,"In ""Everything Is Illuminated,"" young American...",Adventure|Comedy|Drama,Liev Schreiber,Everything Is Illuminated,2005,summary
...,...,...,...,...,...,...,...,...
730,65025,65025|Double Dynamite,"In ""Double Dynamite,"" meek bank teller Johnny ...",Comedy|Musical,Irving Cummings,Double Dynamite,1951,summary
731,65088,65088|Bedtime Stories,"In ""Bedtime Stories,"" Skeeter Bronson, a hotel...",Adventure|Children|Comedy,Adam Shankman,Bedtime Stories,2008,summary
732,65091,65091|Manhattan Melodrama,"In ""Manhattan Melodrama,"" set against the back...",Crime|Drama|Romance,George Cukor,Manhattan Melodrama,1934,summary
733,65126,65126|Choke,"Victor Mancini, a sex addict and Colonial Amer...",Comedy|Drama,Clark Gregg,Choke,2008,summary


In [159]:
last_complete = pd.concat(
    [last_summaries, missings],
    ignore_index=True
)

print(last_complete["doc_type"].value_counts())

doc_type
summary      735
full_plot    735
Name: count, dtype: int64


In [166]:
# last_complete.to_csv('data/last_complete_movies.csv', index=False, encoding='utf-8-sig')
last_complete

,movie_id,source_movie,page_content,genres,director,title,year,doc_type,genre
735,36517,NaN,"Justin Quayle (Ralph Fiennes), a shy, low-leve...",Drama|Thriller,Fernando Meirelles,The Constant Gardener,2005,full_plot,political thriller
0,36517,36517|The Constant Gardener,"In ""The Constant Gardener,"" Justin Quayle, a B...",Drama|Thriller,Fernando Meirelles,The Constant Gardener,2005,summary,NaN
736,36525,NaN,"Elizabeth Masterson (Witherspoon), a young eme...",Comedy|Romance,Mark Waters,Just Like Heaven,2005,full_plot,comedy
1,36525,36525|Just Like Heaven,"In ""Just Like Heaven,"" Elizabeth Masterson, a ...",Comedy|Romance,Mark Waters,Just Like Heaven,2005,summary,NaN
737,36527,NaN,The plot alternates between events immediately...,Drama,John Madden,Proof,2005,full_plot,drama
2,36527,36527|Proof,"In ""Proof"" (2005), Catherine, a mathematician ...",Drama,John Madden,Proof,2005,summary,NaN
738,36529,NaN,"In the early 1980s, Yuri Orlov (Nicolas Cage),...",Action|Crime|Drama|Romance|Thriller|War,Andrew Niccol,Lord of War,2005,full_plot,crime drama
3,36529,36529|Lord of War,"In ""Lord of War,"" set in the early 1980s, Ukra...",Action|Crime|Drama|Romance|Thriller|War,Andrew Niccol,Lord of War,2005,summary,NaN
739,36535,NaN,"Jonathan Safran Foer (Elijah Wood), a young Am...",Adventure|Comedy|Drama,Liev Schreiber,Everything Is Illuminated,2005,full_plot,drama
4,36535,36535|Everything Is Illuminated,"In ""Everything Is Illuminated,"" young American...",Adventure|Comedy|Drama,Liev Schreiber,Everything Is Illuminated,2005,summary,NaN


In [18]:
chroma_clean_df[(chroma_clean_df['doc_type']=='full_plot')].info()

<class 'pandas.core.frame.DataFrame'>
Index: 5498 entries, 0 to 10994
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   chroma_id     5498 non-null   object
 1   movie_id      5498 non-null   int64 
 2   source_movie  25 non-null     object
 3   page_content  5498 non-null   object
 4   genres        5498 non-null   object
 5   director      5498 non-null   object
 6   title         5498 non-null   object
 7   year          5498 non-null   int64 
 8   doc_type      5498 non-null   object
dtypes: int64(2), object(7)
memory usage: 429.5+ KB


In [ ]:
# chroma_clean_df[(chroma_clean_df['title']=='The Constant Gardener') &(chroma_clean_df['doc_type']=='full_plot')  ]['page_content'].values[0]
# # # Επιστρέφει την τιμή της στήλης 'city' στην πρώτη γραμμή που ταιριάζει
# # value = df.query("title == 'The Constant Gardener'")['page_content'].iloc[0]
# print(matrix)

# clean_titles = chroma_clean_df['title']
# last_title  = last_complete['titles']
# last_complete

# matrix = last_complete[last_complete['title']=='The Constant Gardener']['page_content'].to_list()
# # Επιστρέφει την τιμή της στήλης 'city' στην πρώτη γραμμή που ταιριάζει
# matrix = last_complete[last_complete['page_content']=='The Constant Gardener']['page_content'].to_list()
# # Επιστρέφει την τιμή της στήλης 'page_content' στην πρώτη γραμμή που ταιριάζει
# matches = last_complete[last_complete['page_content'].str.contains('The The Constant Gardener', case=False, na=False)]
# pg_content = matches['page_content'].iloc[0] if not matches.empty else None



NameError: name 'last_complete' is not defined

In [276]:
# print(f"type:{type(pg_content)} \n", pg_content)
# pg_content.to_string()
# Επιστρέφει ολόκληρη την πρώτη γραμμή που ικανοποιεί τη συνθήκη
print(matrix)

A woman is cornered by police in an abandoned hotel; after overpowering them with superhuman abilities, a group of sinister superhuman grey green-suited Agents leads the police in a rooftop pursuit. She answers a ringing public telephone and vanishes.
Computer programmer Thomas Anderson, living a double life as the hacker "Neo", feels something is wrong with the world and is puzzled by repeated online encounters with the cryptic phrase "the Matrix". The woman, Trinity, contacts him, saying that a man named Morpheus can explain its meaning; however, the Agents, led by Agent Smith, apprehend Neo and attempt to threaten him into helping them capture the "terrorist" Morpheus. Undeterred, Neo meets Morpheus, who offers him a choice between a red pill that will show him the truth about the Matrix, and a blue pill that will return him to his former life. After swallowing the red pill, his reality disintegrates and Neo awakens, naked, weak and hairless, in a liquid-filled pod, one of countless

In [298]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_movie_plot(plot: str, max_chars: int = 2000) -> list[str]:
    plot = plot.replace("\r\n", "\n").replace("\r", "\n").strip()

    # 1. Split into paragraph-like blocks
    if re.search(r"\n\s*\n", plot):
        paragraphs = re.split(r"\n\s*\n", plot)
    else:
        paragraphs = plot.split("\n")

    paragraphs = [
        p.strip()
        for p in paragraphs
        if p.strip()
    ]

    # 2. Merge small paragraphs until max_chars
    merged_chunks = []
    current = ""

    for paragraph in paragraphs:
        candidate = f"{current}\n\n{paragraph}".strip() if current else paragraph

        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                merged_chunks.append(current)
            current = paragraph

    if current:
        merged_chunks.append(current)

    # 3. Fallback split for oversized chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chars,
        chunk_overlap=300,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    final_chunks = []

    for chunk in merged_chunks:
        if len(chunk) > max_chars:
            final_chunks.extend(splitter.split_text(chunk))
        else:
            final_chunks.append(chunk)

    return final_chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2400,
    chunk_overlap=300,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_text(matrix)



def split_plot_into_paragraphs(plot: str) -> list[str]:
    plot = plot.replace("\r\n", "\n").replace("\r", "\n").strip()

    # Αν υπάρχουν κενές γραμμές, τότε αυτές δείχνουν paragraph breaks
    if re.search(r"\n\s*\n", plot):
        paragraphs = re.split(r"\n\s*\n", plot)
    else:
        # Αλλιώς, κάθε απλό newline θεωρείται paragraph break
        paragraphs = plot.split("\n")

    return [
        p.strip()
        for p in paragraphs
        if p.strip()
    ]

k = split_plot_into_paragraphs(matrix)
l  = split_movie_plot(matrix)

In [ ]:
print(len(k))
print(len(l))
print(l[1])


6
3
As Neo recuperates, Morpheus explains the truth: in the 21st century, intelligent machines waged war against their human creators. When humans blocked the machines' access to solar energy, the machines retaliated by harvesting the humans' bioelectric power. The Matrix is a shared simulation of the world, in which the minds of the harvested humans are trapped and pacified. All free humans live in Zion, the last refuge in the real world. Morpheus and his crew are a group of rebels who hack into the Matrix to "unplug" enslaved humans and recruit them; their understanding of the simulated reality enables them to bend its physical laws, granting them superhuman abilities. Morpheus warns Neo that death within the Matrix also kills the physical body, and that the Agents are powerful sentient programs that eliminate threats to the system. Neo's prowess during virtual combat training lends credence to Morpheus' belief that Neo is "the One", an especially powerful human prophesied to free hu

In [100]:
df = pd.read_csv("data/query_general.csv")
# df[(df['doc_type']=='summary')].info()

df.info()

# df_without_chnks = pd.read_csv("data/without_chunks.csv")
# df[(df['doc_type']=='summary')].info()

# df = pd.read_csv("data/last_summaries_complete.csv")
# df[(df['title']=='The Constant Gardener')]
# df



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10976 entries, 0 to 10975
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   chroma_id     0 non-null      float64
 1   movie_id      10976 non-null  int64  
 2   page_content  10976 non-null  object 
 3   genres        10976 non-null  object 
 4   director      10976 non-null  object 
 5   title         10976 non-null  object 
 6   year          10976 non-null  int64  
 7   doc_type      10976 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 686.1+ KB


In [ ]:
without_titles = df_without_chnks['title']
last_title  = df_last_sum['title']



In [98]:
titles_a = without_titles
titles_b = last_title

unique_a = set(titles_a.dropna().unique())
unique_b = set(titles_b.dropna().unique())

# Τίτλοι που υπάρχουν στο df_a αλλά όχι στο df_b
only_in_a = unique_a - unique_b

# Τίτλοι που υπάρχουν στο df_b αλλά όχι στο df_a
only_in_b = unique_b - unique_a

print("Πόσοι υπάρχουν μόνο στο df_a:", len(only_in_a))
print("Πόσοι υπάρχουν μόνο στο df_b:", len(only_in_b))

print("Μόνο στο df_a:")
print(sorted(only_in_a))

print("Μόνο στο df_b:")
print(sorted(only_in_b))

Πόσοι υπάρχουν μόνο στο df_a: 5308
Πόσοι υπάρχουν μόνο στο df_b: 674
Μόνο στο df_a:
["'Til There Was You", '...And Justice for All', '10', '10 Rillington Place', '10 Things I Hate About You', '10 to Midnight', '101 Dalmatians', '12 Angry Men', '13 Ghosts', '13 Rue Madeleine', '1492: Conquest of Paradise', '1776', '18 Again!', '1941', '1969', '2 Days in the Valley', '2 Fast 2 Furious', '20 Million Miles to Earth', '200 Cigarettes', '2001: A Space Odyssey', '2046', '21 Grams', '24 Hour Party People', '25th Hour', '28 Days', '28 Days Later', '29th Street', '3 Godfathers', '3 Ninjas', '3 Ninjas Kick Back', '3 Ninjas Knuckle Up', '3 Strikes', '3000 Miles to Graceland', '36 Hours', '4 for Texas', '40 Days and 40 Nights', '42nd Street', '48 Hrs.', '50 First Dates', '52 Pick-Up', '54', '633 Squadron', '8 Heads in a Duffel Bag', '8 Mile', '8 Seconds', '84 Charing Cross Road', '9 Songs', 'Abbott and Costello Meet Dr. Jekyll and Mr. Hyde', 'Abbott and Costello Meet Frankenstein', 'Abbott and Cost